In [1]:
import math
import numpy as np
import torch
import torch.nn as nn

In [2]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        scores = q @ k.transpose(-2,-1)
        scores = scores / math.sqrt(q.shape[-1])
        weights = torch.softmax(scores, dim=-1)
        output = weights @ v
        return output


In [3]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.attention = SelfAttention(embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x = x + self.attention(x)
        x = self.norm1(x)
        x = x + self.ffn(x)
        x = self.norm2(x)
        return x
        

In [4]:
class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            [TransformerBlock(embed_dim) for _ in range(num_layers)]
        )
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x